# Lab 01: PyTorch Tensors, Autograd, and Reproducible Training

            **Duration:** 3 hours  
            **Lecture alignment:** Week 1 — Deep-learning computing foundations  
            **CLO mapping:** CLO-1, CLO-2  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Manipulate tensor shapes, dtypes, and broadcasting safely.
- Verify an autograd result numerically.
- Implement and explain a reproducible PyTorch training loop.

            ## Three-hour activity plan

            - 0–25 min: environment, tensor, dtype, and device checks
- 25–65 min: broadcasting and shape-debugging exercises
- 65–105 min: autograd and finite-difference verification
- 105–160 min: build, train, and plot linear regression
- 160–180 min: automated checks and reflection


## Book grounding

            - Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20261
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_01")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_01"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 1, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Will autograd and a centered finite-difference estimate agree exactly? Predict the likely source and scale of any discrepancy.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Tensors, broadcasting, and gradient verification


In [ ]:
a = torch.arange(6, dtype=torch.float32).reshape(2, 3)
b = torch.tensor([1.0, 10.0, 100.0])
broadcast_sum = a + b
assert broadcast_sum.shape == (2, 3)

x0 = torch.tensor(2.0, requires_grad=True)
y0 = x0**3 + 2 * x0
y0.backward()
analytic = x0.grad.item()
eps = 1e-3
finite_difference = (((x0.detach() + eps)**3 + 2*(x0.detach() + eps))
                     - ((x0.detach() - eps)**3 + 2*(x0.detach() - eps))) / (2*eps)
print("Broadcast result:\n", broadcast_sum)
print({"autograd": analytic, "finite_difference": finite_difference.item()})


## Activity 2 — A complete training loop


In [ ]:
n = 160 if FAST_MODE else 1200
x = torch.linspace(-2, 2, n).unsqueeze(1)
y = 3.0 * x + 2.0 + 0.20 * torch.randn_like(x)
loader = DataLoader(TensorDataset(x, y), batch_size=32, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))
model = nn.Linear(1, 1).to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=0.08)
losses = []
for epoch in range(35 if FAST_MODE else 120):
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = F.mse_loss(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    losses.append(total / n)

with torch.no_grad():
    pred = model(x.to(DEVICE)).cpu()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(losses); axes[0].set(title="Training loss", xlabel="epoch", ylabel="MSE")
axes[1].scatter(x, y, s=10, alpha=.45, label="data")
axes[1].plot(x, pred, color="crimson", label="fit"); axes[1].legend()
fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "linear_fit.png", dpi=150); plt.show()
torch.save(model.state_dict(), ARTIFACT_DIR / "linear_model.pt")
print({"weight": model.weight.item(), "bias": model.bias.item(), "final_mse": losses[-1]})


## Automated checks


In [ ]:
assert abs(analytic - finite_difference.item()) < 2e-2
assert losses[-1] < losses[0]
assert abs(model.weight.item() - 3.0) < 0.25
assert (ARTIFACT_DIR / "linear_fit.png").exists()
print("All Lab 01 checks passed.")


## Deliverables

                - Completed tensor and gradient checks
- Loss curve and fitted-line artifact
- Saved state dictionary and explanation of `zero_grad`, `backward`, and `step`

                Submit the executed notebook and the files created in `/content/artifacts/lab_01/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    sizes = [256, 512, 1024]
    timings = {}
    for size in sizes:
        m = torch.randn(size, size, device=DEVICE)
        start = time.perf_counter(); _ = m @ m
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        timings[size] = time.perf_counter() - start
    print(timings)
else:
    print("Extension disabled: benchmark matrix multiplication sizes and device-transfer overhead.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
